In [1]:
!pip install ultralytics opencv-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.0 MB/s eta 0:00:00


In [2]:
from google.colab import files

uploaded = files.upload()

video_path = list(uploaded.keys())[0]
print("Uploaded file:", video_path)

Saving test.mp4 to test (1).mp4
Uploaded file: test (1).mp4


In [3]:
from ultralytics import YOLO

# Load pretrained YOLOv8 model (auto-downloads)
model = YOLO("yolov8n.pt")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [4]:
import cv2

# Open video
cap = cv2.VideoCapture(video_path)

# Get video properties
frame_width = int(cap.get(3))
frame_height = int(cap.get(4))
fps = int(cap.get(cv2.CAP_PROP_FPS))

# Output file
output_path = "output.avi"
fourcc = cv2.VideoWriter_fourcc(*'XVID')
out = cv2.VideoWriter(output_path, fourcc, fps, (frame_width, frame_height))

# COCO class IDs for:
# person = 0, car = 2, motorcycle = 3, bus = 5, truck = 7
target_classes = [0, 2, 3, 5, 7]

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Run YOLO detection
    results = model(frame)

    for result in results:
        boxes = result.boxes

        for box in boxes:
            cls = int(box.cls[0])

            if cls in target_classes:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                conf = float(box.conf[0])

                label = model.names[cls]

                # Draw bounding box
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(frame, f"{label} {conf:.2f}",
                            (x1, y1 - 10),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            0.5, (0, 255, 0), 2)

    out.write(frame)

cap.release()
out.release()

print("Processing complete. Output saved as:", output_path)


0: 384x640 11 persons, 1 potted plant, 420.4ms
Speed: 19.6ms preprocess, 420.4ms inference, 47.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 10 persons, 1 potted plant, 203.2ms
Speed: 9.3ms preprocess, 203.2ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 11 persons, 1 potted plant, 200.4ms
Speed: 6.9ms preprocess, 200.4ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 11 persons, 1 potted plant, 214.6ms
Speed: 8.3ms preprocess, 214.6ms inference, 2.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 10 persons, 1 bicycle, 1 potted plant, 209.7ms
Speed: 6.4ms preprocess, 209.7ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 10 persons, 1 bicycle, 1 potted plant, 208.3ms
Speed: 6.6ms preprocess, 208.3ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 10 persons, 1 bicycle, 1 potted plant, 216.5ms
Speed: 11.1ms preprocess, 216.5ms inferen

In [5]:
from google.colab import files

files.download("output.avi")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>